# An-Ra Unified Colab Trainer

Single end-to-end workflow. No Gemini/API keys required.


In [ ]:
# Cell 1: Environment + Drive
import os, sys, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable T4 GPU in Colab Runtime settings')

deps = ['torch', 'fastapi', 'uvicorn', 'httpx', 'sympy', 'psutil', 'sentence-transformers']
for dep in deps:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', dep], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/KellerJordan/Muon'], check=False)


In [ ]:
# Cell 2: Clone repo + path bootstrap
REPO = Path('/content/An-Ra')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print('Repo ready:', REPO)


In [ ]:
# Cell 3: Health check
import importlib
mods = ['generate', 'app', 'symbolic_bridge', 'ouroboros_numpy', 'sovereignty_bridge', 'identity_injector']
for m in mods:
    try:
        importlib.import_module(m)
        print('OK', m)
    except Exception as e:
        print('WARN', m, e)


In [ ]:
# Cell 4: 30-minute resumable training session (single command)
# Auto-uses your dataset in training/training_data and resumes from local/Drive checkpoint.
cmd = [
    sys.executable, '-m', 'training.train_unified',
    '--mode', 'session',
    '--session_minutes', '30',
    '--checkpoint_path', 'anra_brain.pt',
    '--ouroboros_steps', '5000'
]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('exit=', proc.returncode)


In [ ]:
# Cell 5B: 30-minute direct base-training routine (fallback)
import shutil
DRIVE = Path('/content/drive/MyDrive/AnRa')
for name in ['anra_ouroboros.pt', 'anra_brain_identity.pt', 'anra_brain.pt', 'tokenizer.pkl']:
    src = DRIVE / name
    dst = REPO / name
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('restored', name)

result = subprocess.run([
    sys.executable, 'scripts/build_brain.py',
    '--data_path', 'training_data/anra_dataset_v6_1.txt',
    '--checkpoint_path', 'anra_brain.pt',
    '--max_minutes', '30'
], cwd=str(REPO), check=False)
print('session training exit=', result.returncode)


In [ ]:
# Cell 6: Verification
subprocess.run([sys.executable, 'scripts/verify_structure.py'], cwd=str(REPO), check=False)
subprocess.run([sys.executable, '-m', 'tests.test_suite'], cwd=str(REPO), check=False)
subprocess.run([sys.executable, 'anra.py', '--phase3-status'], cwd=str(REPO), check=False)
